<a href="https://colab.research.google.com/github/mamontoff2018-lgtm/BerezinaAlg/blob/main/practices/LR2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ЛАБОРАТОРНАЯ РАБОТА №2. Классификация методом k-ближайших соседей (kNN)

## Задания


In [1]:
import numpy as np
from collections import Counter
from sklearn import datasets
from sklearn.model_selection import train_test_split

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        return np.array([self._predict(x) for x in X])

    def _predict(self, x):
        distances = np.sum((self.X_train - x)**2, axis=1)

        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = self.y_train[k_indices]

        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

#ЗАДАНИЕ 1
data = datasets.load_wine()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

x_min = X_train.min(axis=0)
x_max = X_train.max(axis=0)

# Нормализация в диапазон [0, 1] (min-max scaling)
X_train_norm = (X_train - x_min) / (x_max - x_min)
X_test_norm  = (X_test  - x_min) / (x_max - x_min)

print("Обучающая выборка (нормализованная) – первые 3 строки:\n", X_train_norm[:3])
print("\nФорма данных:", X_train_norm.shape)

# ЗАДАНИЕ 2
clf = KNN(k=5)
clf.fit(X_train_norm, y_train)

print("\nМодель обучена на {} объектах.".format(len(X_train_norm)))

# ЗАДАНИЕ 3
predictions = clf.predict(X_test_norm)

accuracy = np.mean(predictions == y_test)
print(f"\nТочность модели: {accuracy * 100:.2f}%")

# Выводим метки для сравнения
print("\nРеальные метки:    ", y_test)
print("Предсказанные:     ", predictions)

# Индексы ошибок
errors = np.where(predictions != y_test)[0]
print("\nИндексы ошибок:", errors)

# точность для каждого класса
for class_label in np.unique(y_test):
    mask = y_test == class_label
    class_acc = np.mean(predictions[mask] == y_test[mask])
    print(f"  Класс {class_label}: точность {class_acc * 100:.2f}%")

Обучающая выборка (нормализованная) – первые 3 строки:
 [[0.87105263 0.16089613 0.71657754 0.74226804 0.30434783 0.62758621
  0.20464135 0.75471698 0.72151899 1.         0.07317073 0.25274725
  0.30102443]
 [0.39473684 0.94093686 0.68449198 0.74226804 0.2826087  0.27931034
  0.05485232 0.94339623 0.21518987 0.28952043 0.27642276 0.15384615
  0.18676123]
 [0.35263158 0.03665988 0.39572193 0.40721649 0.19565217 0.87586207
  0.71940928 0.20754717 0.48417722 0.24511545 0.45528455 0.54945055
  0.30102443]]

Форма данных: (142, 13)

Модель обучена на 142 объектах.

Точность модели: 94.44%

Реальные метки:     [0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]
Предсказанные:      [0 0 2 0 1 0 1 2 1 2 0 2 0 2 0 1 1 1 0 1 0 1 1 2 2 2 1 0 1 0 0 1 2 0 0 0]

Индексы ошибок: [13 27]
  Класс 0: точность 100.00%
  Класс 1: точность 85.71%
  Класс 2: точность 100.00%


## Контрольные вопросы

1. Зачем нужна нормализация? Что произойдет с расстоянием между точками, если один признак измеряется в тысячах, а другой в единицах?

    Нормализация приводит все признаки к единому масштабу (например, [0, 1]). Если этого не сделать, признак с большими абсолютными значениями (например, доход в тысячах рублей) будет доминировать в расчете Евклидова расстояния, полностью "подавляя" вклад признака с малыми значениями (например, возраст в годах). В результате расстояние будет определяться практически одним признаком, а остальные факторы модель не учтет.

2. Если данные сильно зашумлены (много случайных выбросов), стоит ли увеличивать или уменьшать k? Почему?

    При сильной зашумленности данных значение k лучше увеличить (например, с 3 до 7–11). Чем больше соседей участвует в голосовании, тем больше усредняется их влияние, и единичные шумовые объекты меньше искажают результат. Малое k (например, k=1) делает модель чрезмерно чувствительной к каждому выбросу, что ведёт к переобучению. Однако слишком большое k размывает границы классов — нужен баланс.

3. Почему kNN называют «ленивым» алгоритмом обучения (lazy learner)?

    Потому что в фазе "обучения" (метод fit) модель вообще не строит никаких обобщений и не вычисляет параметры. Она просто запоминает всю обучающую выборку. Основная вычислительная работа (расчёт расстояний до всех точек и поиск ближайших соседей) выполняется только при поступлении тестового объекта, то есть во время предсказания. Поэтому алгоритм — "ленивый" (откладывает работу на потом).

4. В каких случаях Евклидово расстояние может быть плохой метрикой для kNN?

    Евклидово расстояние плохо работает, когда:

    Признаки имеют разную природу и масштаб (проблема решается нормализацией, но не до конца).

    Данные высокоразмерны (много признаков) — из-за "проклятия размерности" все точки становятся почти равноудалёнными, и понятие ближайшего соседа теряет смысл.

    Присутствуют категориальные признаки (цвет, страна) — для них нужно использовать другие метрики (например, расстояние Хэмминга).

    Данные имеют сложную структуру (например, классы в виде концентрических окружностей) — Евклидово расстояние не отражает реальную близость по смыслу.

    Если важна не величина различия, а направление векторов признаков — здесь лучше подходит косинусное расстояние.